# 08 Appendix Additional Ablations

This notebook collects the non-winning branches that still matter scientifically. They should stay out of the main text, but they are important for showing that the final thesis claims are not based on a shallow search.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if not SRC.exists():
    raise FileNotFoundError(f"Expected thesis src at {SRC}; run notebooks from thesis/notebooks")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()

fig_dir, table_dir = notebook_output_dirs("08_appendix_additional_ablations")
import json
manifest = json.loads((ROOT / "src/qc_thesis/modeling/PROVENANCE_MANIFEST.json").read_text())
appendix_files = pd.DataFrame(
    [{"group": group, "path": path} for group, paths in manifest.items() for path in paths if "appendix_variants" in path]
)
fpos_groups = get_recipe_groups("fpos")
fmiss_groups = get_recipe_groups("fmiss")


## 0. Full recipe inventory: every hypothesis tested

The appendix begins with the complete recipe registry. This is the authoritative map of every modeling hypothesis that was registered, run, and archived — including the dead ends that informed the thesis claims.

In [ ]:
PROTOCOL = "recording_disjoint_main"

inventory = build_recipe_inventory()
display(Markdown(f"**{len(inventory)} total recipes registered** across `fpos` and `fmiss`"))
display(inventory)
save_table(inventory, table_dir, "full_recipe_inventory")

# Load every registered recipe and build a combined benchmark table
all_results = {}
for _, row in inventory.iterrows():
    try:
        result = load_or_run_experiment(row["recipe_id"], row["target"], PROTOCOL)
        all_results[row["recipe_id"]] = result
    except Exception as exc:
        print(f"  skipped {row['recipe_id']}: {exc}")

def _m(result):
    r = result.metrics.iloc[0] if not result.metrics.empty else {}
    return {"r2": float(r.get("r2", float("nan"))), "mae": float(r.get("mae", float("nan")))}

full_benchmark = pd.DataFrame([
    {"recipe_id": rid, "target": res.recipe.target,
     "label": res.recipe.label, "ablation_group": res.recipe.ablation_group,
     "model_family": res.recipe.model_family, "main_text": res.recipe.main_text,
     **_m(res)}
    for rid, res in all_results.items()
]).sort_values(["target", "r2"], ascending=[True, False]).reset_index(drop=True)

display(full_benchmark)
save_table(full_benchmark, table_dir, "full_benchmark_all_recipes")

In [ ]:
for target in ["fpos", "fmiss"]:
    target_bench = full_benchmark[full_benchmark["target"] == target].copy()
    display(Markdown(f"### Full `{target}` benchmark ({len(target_bench)} recipes)"))
    display(target_bench[["label", "ablation_group", "model_family", "r2", "mae", "main_text"]])
    fig, _ = plot_benchmark_metric(target_bench.rename(columns={"recipe_id": "recipe_id"}),
                                   metric="r2", title=f"`{target}` all recipes by R²")
    save_figure(fig, fig_dir, f"{target}_full_recipe_r2")
    display(fig)
    plt.close(fig)


## 1. Appendix provenance inventory

These files remain in the thesis package because they support negative results, appendix ablations, or methodological cautionary notes.


In [ ]:
display(appendix_files)
save_table(appendix_files, table_dir, "appendix_provenance_inventory")


## 2. Ablation grouping reference

The appendix should still be organized by scientific question, not by raw script name. These group maps are the compact reference for the appendix chapter structure.


In [ ]:
appendix_fpos = pd.DataFrame([(k, ', '.join(v)) for k, v in fpos_groups.items()], columns=["ablation_group", "recipe_ids"])
appendix_fmiss = pd.DataFrame([(k, ', '.join(v)) for k, v in fmiss_groups.items()], columns=["ablation_group", "recipe_ids"])
display(appendix_fpos)
display(appendix_fmiss)
save_table(appendix_fpos, table_dir, "appendix_fpos_recipe_groups")
save_table(appendix_fmiss, table_dir, "appendix_fmiss_recipe_groups")


## 3. What belongs in appendix rather than main text

The scientific rule is simple:

- main text keeps the strongest benchmark ladder, target asymmetry, and robustness story
- appendix keeps negative branches, tuning branches, transport/synthetic branches, and exploratory dead ends that still matter for scientific completeness


In [ ]:
display(Markdown(
    """
**Appendix families to emphasize**

- synthetic augmentation and paired-conditioned transport
- residual correction and manifold-context variants
- waveform enrichment and multiscale waveform variants
- Bayesian tuning and meta/bagging branches
- tree interaction and paper-inspired transfer branches
"""
))
